# 第 58 课｜从零实现语言模型 DPO
## 参考模型、成对批次与 LoRA 更新

**把上一课的四组 log 概率，接回真正会逐字符生成的 Transformer。**

本课完成：**SFT 起点 → 固定参考模型 → 注入 Q/V LoRA → 成对偏好训练 → 开发集选轮 → 留出生成 → 保存重载。**

| 目标 | 验收方式 |
|---|---|
| 构造 chosen / rejected 成对输入 | 相同提示、同位标签、仅回答和 EOS 参与概率和 |
| 实现标准 sigmoid-DPO | 核对损失、梯度、参考差值与 beta |
| 只更新策略 adapter | 检查参数集合、基座哈希、参考梯度 |
| 区分偏好分数与真正生成 | 同时报告成对排序、目标概率、自由生成与退步题 |
| 保存可复核状态 | 干净 SFT 基座重载 adapter，核对 logits 与编号 |

**范围：PyTorch + 标准库，完全离线。** 这是从随机参数训练的小型语言模型，不下载 Qwen，不使用 TRL/PEFT，不实现量化、混合精度或 PPO/GRPO。合成任务成绩不代表自然语言推理能力。

## 0. 怎样运行

沿用已有 PyTorch 的 Jupyter 内核，从上到下执行。较长单元是完整实现，短单元用于观察与修改。

**Notebook 内嵌全部代码，不依赖同名 `.py` 或前几课文件。** 脚本也可以单独运行：

```bash
python lesson_58_language_model_dpo_and_lora.py --checks-only
python lesson_58_language_model_dpo_and_lora.py
```

默认 CPU、FP32，数学与梯度核对使用 FP64。练习开关默认关闭；“运行全部成功”不代表已经完成练习。建议分两次完成：先看数据、损失与一轮梯度，再看完整训练和生成评估。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第58课：从零实现语言模型 DPO——参考模型、成对批次与 LoRA 更新。

Python 3.10+，PyTorch，完全离线。默认在CPU训练真正的字符自回归Transformer。
先用训练侧做SFT，再复制固定参考模型，只更新策略的Q/V LoRA。
小型合成任务，不是预训练大模型效果、TRL/PEFT兼容性或3060性能评测。
"""
from __future__ import annotations


import argparse
import copy
import hashlib
import json
import math
import platform
import random
import tempfile
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要已有的PyTorch环境；请选择与脚本相同环境的Jupyter内核。") from exc

VERSION = "lesson58-v1"
PAD, BOS, EOS, IGNORE = 0, 1, 2, -100
SOURCES = {
    "DPO": "https://arxiv.org/abs/2305.18290",
    "DPO_implementation": "https://github.com/eric-mitchell/direct-preference-optimization",
    "TRL": "https://huggingface.co/docs/trl/dpo_trainer",
    "LoRA": "https://arxiv.org/abs/2106.09685",
    "log_softmax": "https://docs.pytorch.org/docs/2.10/generated/torch.nn.functional.log_softmax.html",
    "autograd": "https://docs.pytorch.org/docs/2.10/notes/autograd.html",
}


class LessonError(ValueError):
    """教学契约未满足；不能通过静默截断或默认填充值继续训练。"""


def require(ok: bool, message: str) -> None:
    if not ok:
        raise LessonError(message)


def canonical(value) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def digest(value) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def write_json(path: Path, value) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True,
                               indent=2, allow_nan=False) + "\n", encoding="utf-8")


def tensor_hash(values: dict[str, torch.Tensor]) -> str:
    h = hashlib.sha256()
    for name, tensor in sorted(values.items()):
        t = tensor.detach().cpu().contiguous()
        h.update(canonical([name, list(t.shape), str(t.dtype)]).encode())
        h.update(bytes(t.reshape(-1).view(torch.uint8).tolist()))
    return h.hexdigest()


def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device(name: str) -> torch.device:
    require(name in {"cpu", "cuda", "auto"}, "device仅支持cpu/cuda/auto")
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    if name == "cuda" and not torch.cuda.is_available():
        raise LessonError("明确请求CUDA，但当前PyTorch没有可用CUDA；不会悄悄回退。")
    return torch.device(name)


@dataclass(frozen=True)
class ModelConfig:
    vocab_size: int
    dim: int = 48
    heads: int = 4
    layers: int = 2
    ff_dim: int = 96
    max_len: int = 32

    def validate(self) -> None:
        for value in asdict(self).values():
            require(type(value) is int and value > 0, "模型配置需要正整数")
        require(self.dim % self.heads == 0, "隐藏维度须整除头数")
        require(self.vocab_size > 3 and self.max_len >= 3, "词表或长度过小")


@dataclass(frozen=True)
class TrainConfig:
    seed: int = 58
    sft_steps: int = 160
    sft_lr: float = 0.003
    dpo_epochs: int = 8
    dpo_lr: float = 0.001
    batch_size: int = 16  # 偏好对数，实际模型前向有2B条序列。
    rank: int = 4
    alpha: float = 8.0
    beta: float = 0.1
    max_grad_norm: float = 1.0
    cached_reference: bool = False

    def validate(self) -> None:
        for key in ["sft_steps", "dpo_epochs", "batch_size", "rank"]:
            require(type(getattr(self, key)) is int and getattr(self, key) > 0,
                    key + "必须为正整数")
        require(type(self.seed) is int and self.seed >= 0, "seed必须非负")
        for key in ["sft_lr", "dpo_lr", "alpha", "beta", "max_grad_norm"]:
            v = getattr(self, key)
            require(type(v) in (int, float) and math.isfinite(v) and v > 0,
                    key + "必须有限且为正")
        require(type(self.cached_reference) is bool, "cached_reference必须为bool")

In [2]:
torch.set_num_threads(1)
seed_all(58)
DEVICE = "cpu"
OUTPUT_ROOT = Path("lesson_58_outputs")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("实际实验设备:", DEVICE, "；CUDA可用:", torch.cuda.is_available())
print("当前目录:", Path.cwd())

Python: 3.13.5
PyTorch: 2.10.0+cpu
实际实验设备: cpu ；CUDA可用: False
当前目录: /mnt/data/lesson58_work/notebook_run


## 1. 本课数据：可验证的简短复制任务

例如：

```text
提示：last:3,12=
较好回答：12
较差回答：1
```

任务是输出最后一个整数。偏好标签来自显式规则，不是人类标注，也不是从预训练模型采样得到。

两个输入整数都取自 `-3…12`，一共 256 个不同组合。固定种子先打乱，再分为 **160 训练、48 开发、48 测试**。每个组合是一个来源；三侧不共享完整提示，但共享语法和整数范围。

SFT 只使用训练侧 chosen，DPO 只使用训练侧偏好对，开发集选择轮次，测试集只在选择落盘以后使用。字符词表预先固定为任务语法，不从测试答案扩充。

这种留出只能检查新组合，**不是未见语言、未见数学规律或真实应用泛化。**

In [3]:
class CharCodec:
    """固定任务字符表：不是从测试集学习出来的真实BPE；未知字符直接拒绝。"""
    def __init__(self):
        self.tokens = ["<pad>", "<bos>", "<eos>"] + list("last:0123456789-,=")
        require(len(set(self.tokens)) == len(self.tokens), "字符表不能重复")
        self.to_id = {s: i for i, s in enumerate(self.tokens)}

    def encode(self, text: str) -> list[int]:
        require(type(text) is str, "text必须为字符串")
        require(all(ch in self.to_id for ch in text), "字符超出固定任务词表")
        return [self.to_id[ch] for ch in text]

    def decode(self, ids: list[int]) -> str:
        require(all(type(i) is int and 0 <= i < len(self.tokens) for i in ids),
                "token编号非法")
        return "".join(self.tokens[i] for i in ids)

    def contract(self) -> dict:
        return {"tokens": self.tokens, "special_ids": [PAD, BOS, EOS],
                "normalization": "none", "unknown": "error", "version": VERSION}


def make_splits(seed: int = 58) -> dict[str, list[dict]]:
    """任务：复制last:a,b=中最后一个整数b。不做人类偏好或自然语言能力声明。"""
    numbers = list(range(-3, 13))
    all_rows = []
    for a in numbers:
        for b in numbers:
            rejected = numbers[(numbers.index(b) + 5) % len(numbers)]
            all_rows.append({
                "id": f"pair_{a}_{b}", "group_id": f"source_{a}_{b}",
                "prompt": f"last:{a},{b}=", "chosen": str(b),
                "rejected": str(rejected), "expected": str(b),
                "label_source": "explicit_last_integer_rule_v1",
            })
    random.Random(seed).shuffle(all_rows)
    split = {"train": all_rows[:160], "dev": all_rows[160:208], "test": all_rows[208:]}
    for name, rows in split.items():
        for row in rows:
            row["split"] = name
    return split


def validate_splits(splits: dict) -> None:
    require(set(splits) == {"train", "dev", "test"}, "划分名称不完整")
    ids, groups, prompts = set(), set(), set()
    for part, rows in splits.items():
        require(bool(rows), "每一侧都必须非空")
        for row in rows:
            require(row["split"] == part, "记录与划分不匹配")
            require(row["id"] not in ids, "重复题目ID")
            require(row["group_id"] not in groups, "来源跨侧或重复")
            require(row["prompt"] not in prompts, "提示重复")
            require(row["chosen"] == row["expected"] != row["rejected"], "规则标签错误")
            ids.add(row["id"]); groups.add(row["group_id"]); prompts.add(row["prompt"])


def encode_completion(codec: CharCodec, prompt: str, answer: str,
                      max_len: int) -> tuple[list[int], list[int]]:
    require(type(prompt) is str and bool(prompt), "提示不能为空")
    require(type(answer) is str and bool(answer), "回答不能为空")
    prefix = [BOS] + codec.encode(prompt)
    target = codec.encode(answer) + [EOS]
    ids = prefix + target
    require(len(ids) <= max_len, "完整回答越过max_len；拒绝静默截断偏好对")
    labels = [IGNORE] * len(prefix) + target
    return ids, labels


def collate_pairs(rows: list[dict], codec: CharCodec, max_len: int,
                  device: str | torch.device = "cpu") -> dict:
    """前B行为chosen，后B行为rejected；每一对共享完全相同的prompt前缀。"""
    require(bool(rows), "不能整理空批次")
    require(len({r['id'] for r in rows}) == len(rows), "一个批次不能有重复ID")
    sequences, targets = [], []
    for branch in ["chosen", "rejected"]:
        for row in rows:
            require(row["chosen"] != row["rejected"], "不接受相同completion组成的偏好对")
            ids, labels = encode_completion(codec, row["prompt"], row[branch], max_len)
            sequences.append(ids); targets.append(labels)
    width = max(map(len, sequences))
    input_ids = torch.full((2 * len(rows), width), PAD, dtype=torch.long, device=device)
    labels = torch.full_like(input_ids, IGNORE)
    for i, (seq, target) in enumerate(zip(sequences, targets)):
        input_ids[i, :len(seq)] = torch.tensor(seq, device=device)
        labels[i, :len(seq)] = torch.tensor(target, device=device)
    return {"input_ids": input_ids, "labels": labels,
            "attention_mask": input_ids.ne(PAD), "pair_ids": [r["id"] for r in rows],
            "pair_count": len(rows)}

In [4]:
codec = CharCodec()
splits = make_splits()
validate_splits(splits)
print("划分:", {k: len(v) for k, v in splits.items()})
print("词表大小:", len(codec.tokens))
print("一条训练记录:", json.dumps(splits["train"][0], ensure_ascii=False, indent=2))

划分: {'train': 160, 'dev': 48, 'test': 48}
词表大小: 21
一条训练记录: {
  "id": "pair_-1_0",
  "group_id": "source_-1_0",
  "prompt": "last:-1,0=",
  "chosen": "0",
  "rejected": "5",
  "expected": "0",
  "label_source": "explicit_last_integer_rule_v1",
  "split": "train"
}


## 2. 同位标签：提示保留为输入，回答才参与损失

对于 `last:3,12=` 和回答 `12`：

```text
输入：  BOS  l a s t : 3 , 1 2 =  1 2 EOS
标签： -100 …………………… -100    1 2 EOS
```

`=` 位置的 logits 预测第一个回答字符 `1`。`1` 位置预测 `2`，`2` 位置预测 EOS。

**本课模型不接收 labels、不在内部计算损失。** 因此我们在外部用 `logits[:, :-1]` 对齐 `labels[:, 1:]`，只移位一次。迁移到会内部移位的 Hugging Face CausalLM 时，不要重复移位。

EOS 是训练目标，PAD 不是。只屏蔽提示的直接损失，不等于阻止模型读取提示，也不等于提示表示没有间接梯度。

In [5]:
demo_rows = [
    {"id":"demo1", "prompt":"last:3,12=", "chosen":"12", "rejected":"-3"},
    {"id":"demo2", "prompt":"last:1,2=", "chosen":"2", "rejected":"12"},
]
demo_batch = collate_pairs(demo_rows, codec, max_len=32)
print("input_ids:", tuple(demo_batch["input_ids"].shape))
print("labels:\n", demo_batch["labels"])
for row_index, (ids, labels) in enumerate(zip(demo_batch["input_ids"], demo_batch["labels"])):
    pieces = [f"{codec.tokens[i]}{'*' if lab != IGNORE else ''}" for i,lab in zip(ids.tolist(),labels.tolist())]
    print(row_index, " ".join(pieces))
print("* 表示该位置的token作为目标参与损失；由前一个位置的logits预测。")

input_ids: (4, 14)
labels:
 tensor([[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,    9,
           10,    2],
        [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100,   10,    2,
         -100, -100],
        [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,   18,
           11,    2],
        [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100,    9,   10,
            2, -100]])
0 <bos> l a s t : 3 , 1 2 = 1* 2* <eos>*
1 <bos> l a s t : 1 , 2 = 2* <eos>* <pad> <pad>
2 <bos> l a s t : 3 , 1 2 = -* 3* <eos>*
3 <bos> l a s t : 1 , 2 = 1* 2* <eos>* <pad>
* 表示该位置的token作为目标参与损失；由前一个位置的logits预测。


### 2.1 为什么一个偏好 batch 是 2B 条序列？

排列约定是：**前 B 行全部 chosen，后 B 行全部 rejected**，不是 c0/r0/c1/r1 交错。

| 张量 | 形状 |
|---|---|
| 输入与标签 | `[2B,T]` |
| logits | `[2B,T,V]` |
| 每条 completion 的 log 概率 | `[2B]` |
| 拆出的 chosen / rejected | 各 `[B]` |
| 每对 DPO loss | `[B]` |

两侧统一右补齐到同一 T，但每一对的提示必须完全相同。生产 BPE/Chat Template 可能有跨边界 token，不能把这里的字符拼接规则直接照搬；应固定模板并验证边界。参考：[TRL DPO 数据与实现说明](https://huggingface.co/docs/trl/dpo_trainer)。

## 3. 训练的是完整的因果语言模型，不是概率表

结构：

```text
Token + 绝对位置 Embedding
    → 2个 Pre-LN 因果 Transformer Block
    → 最终 LayerNorm
    → 词表输出头
```

默认隐藏维度 48、4 个注意力头、FFN 中间维度 96、长度上限 32、21 个 token。所有 Dropout 都是 0。

为聚焦 DPO，使用已经学过的简化组件，而不是在本课再次加入 RoPE、GQA、SwiGLU 或 KV cache。因果约束与 PAD key mask 仍要正确。

SFT 阶段更新全部模型参数；DPO 阶段冻结它们，只在每层 Q/V 加入 rank=4、alpha=8 的 LoRA。LoRA 改变可训练参数集合，不改变 DPO 的定义。[LoRA 论文](https://arxiv.org/abs/2106.09685)

In [6]:
class CausalAttention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.heads, self.head_dim = cfg.heads, cfg.dim // cfg.heads
        self.q_proj = nn.Linear(cfg.dim, cfg.dim, bias=False)
        self.k_proj = nn.Linear(cfg.dim, cfg.dim, bias=False)
        self.v_proj = nn.Linear(cfg.dim, cfg.dim, bias=False)
        self.o_proj = nn.Linear(cfg.dim, cfg.dim, bias=False)

    def forward(self, x: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
        b, t, d = x.shape
        def heads(z):
            return z.reshape(b, t, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = (heads(layer(x)) for layer in [self.q_proj, self.k_proj, self.v_proj])
        causal = torch.ones(t, t, dtype=torch.bool, device=x.device).tril()
        allowed = causal[None, None, :, :] & valid[:, None, None, :]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        weights = F.softmax(scores.masked_fill(~allowed, -torch.inf), dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().reshape(b, t, d)
        return self.o_proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.dim)
        self.attn = CausalAttention(cfg)
        self.norm2 = nn.LayerNorm(cfg.dim)
        self.ffn = nn.Sequential(nn.Linear(cfg.dim, cfg.ff_dim), nn.GELU(),
                                 nn.Linear(cfg.ff_dim, cfg.dim))

    def forward(self, x: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x), valid)
        return x + self.ffn(self.norm2(x))


class TinyLM(nn.Module):
    """两层因果Transformer，绝对位置、Pre-LN、GELU；所有Dropout均为零。

    为聚焦DPO故意不添加RoPE、GQA、KV cache或量化；不是生产模型缩小复制品。
    forward只返回logits，由外部损失移位一次。输入必须从BOS开始、右侧补齐。
    """
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        cfg.validate()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.dim, padding_idx=PAD)
        self.position_embedding = nn.Embedding(cfg.max_len, cfg.dim)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm = nn.LayerNorm(cfg.dim)
        self.head = nn.Linear(cfg.dim, cfg.vocab_size, bias=False)
        self.apply(self._init_weights)
        with torch.no_grad():
            self.token_embedding.weight[PAD].zero_()

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        require(ids.ndim == 2 and ids.dtype == torch.long, "input_ids需要[B,T] long")
        require(ids.shape[0] > 0 and 0 < ids.shape[1] <= self.cfg.max_len, "输入长度非法")
        require(bool(((ids >= 0) & (ids < self.cfg.vocab_size)).all()), "token越界")
        require(bool((ids[:, 0] == BOS).all()), "本课每条输入从BOS开始，拒绝全PAD")
        valid = ids.ne(PAD)
        require(not bool((valid[:, 1:] & ~valid[:, :-1]).any()), "本课只支持右补齐")
        positions = torch.arange(ids.shape[1], device=ids.device)
        x = self.token_embedding(ids) + self.position_embedding(positions)[None, :, :]
        for block in self.blocks:
            x = block(x, valid)
        return self.head(self.norm(x))


class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: float):
        super().__init__()
        require(type(base) is nn.Linear, "只接受未注入的Linear")
        require(type(rank) is int and 0 < rank <= min(base.in_features, base.out_features),
                "rank超出本课允许范围")
        require(math.isfinite(alpha) and alpha > 0, "alpha必须有限为正")
        self.base = base
        self.base.requires_grad_(False)
        self.rank, self.alpha, self.scaling = rank, float(alpha), float(alpha) / rank
        self.enabled = True
        self.lora_A = nn.Parameter(base.weight.new_empty(rank, base.in_features))
        self.lora_B = nn.Parameter(base.weight.new_zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.base(x)
        if self.enabled:
            out = out + self.scaling * F.linear(F.linear(x, self.lora_A), self.lora_B)
        return out


def is_adapter(name: str) -> bool:
    return name.endswith(".lora_A") or name.endswith(".lora_B")


def inject_lora(model: TinyLM, rank: int = 4, alpha: float = 8.0) -> TinyLM:
    require(not any(isinstance(m, LoRALinear) for m in model.modules()), "拒绝重复注入")
    model.requires_grad_(False)
    # SFT刚结束可能留有.grad；清掉它们，才能审计DPO阶段是否产生新梯度。
    for p in model.parameters():
        p.grad = None
    for block in model.blocks:
        block.attn.q_proj = LoRALinear(block.attn.q_proj, rank, alpha)
        block.attn.v_proj = LoRALinear(block.attn.v_proj, rank, alpha)
    require(all(p.requires_grad == is_adapter(n) for n, p in model.named_parameters()),
            "可训练集合必须恰好等于Q/V的A/B")
    return model


def freeze_reference(base: TinyLM) -> TinyLM:
    ref = copy.deepcopy(base).eval().requires_grad_(False)
    for p in ref.parameters():
        p.grad = None
    return ref


def base_state(model: TinyLM) -> dict[str, torch.Tensor]:
    return {name.replace(".base.", "."): value
            for name, value in model.state_dict().items() if not is_adapter(name)}


def adapter_snapshot(model: TinyLM) -> dict[str, torch.Tensor]:
    return {name: p.detach().cpu().clone() for name, p in model.named_parameters()
            if is_adapter(name)}


def load_adapter(model: TinyLM, state: dict) -> None:
    expected = {n for n, _ in model.named_parameters() if is_adapter(n)}
    require(set(state) == expected, "adapter键集合不匹配")
    # 先检查全部张量，再修改，避免部分加载。
    params = dict(model.named_parameters())
    for name, value in state.items():
        require(isinstance(value, torch.Tensor) and value.shape == params[name].shape,
                "adapter形状不匹配")
        require(value.is_floating_point() and bool(torch.isfinite(value).all()), "adapter非法数值")
    with torch.no_grad():
        for name, value in state.items():
            params[name].copy_(value.to(params[name]))


@contextmanager
def adapter_disabled(model: TinyLM):
    layers = [m for m in model.modules() if isinstance(m, LoRALinear)]
    previous = [m.enabled for m in layers]
    try:
        for layer in layers:
            layer.enabled = False
        yield
    finally:
        for layer, enabled in zip(layers, previous):
            layer.enabled = enabled

In [7]:
seed_all(580)
model_config = ModelConfig(vocab_size=len(codec.tokens))
probe_base = TinyLM(model_config).eval()
probe_reference = freeze_reference(probe_base)
probe_policy = inject_lora(copy.deepcopy(probe_base), rank=4, alpha=8)
with torch.no_grad():
    plain = probe_reference(demo_batch["input_ids"])
    adapted = probe_policy(demo_batch["input_ids"])
print("logits:", tuple(plain.shape))
print("基座参数:", sum(p.numel() for p in probe_reference.parameters()))
print("LoRA参数:", sum(p.numel() for p in probe_policy.parameters() if p.requires_grad))
print("初始输出差异:", (plain-adapted).abs().max().item())

logits: (4, 14, 21)
基座参数: 41184
LoRA参数: 1536
初始输出差异: 0.0


## 4. 参考模型：冻结训练开始时的策略，而不是每步跟着改

完整实验先做 SFT，再从**同一个 SFT 检查点**建立：

```text
固定参考模型 πref：独立副本，eval()，requires_grad_(False)，no_grad前向
当前策略 πθ：    同样的SFT基座＋可训练Q/V LoRA
```

A 随机、B 为零，因此初始策略和参考的输出相同。参考模型不进入优化器。

`eval()` 只影响 Dropout 等模块模式，不会关闭自动求导；`requires_grad_(False)` 冻结参数，`no_grad()` 关闭这段运算的图记录，三者不是同义词。[PyTorch 自动求导说明](https://docs.pytorch.org/docs/2.10/notes/autograd.html)

**不要使用 `reference = policy`。** 如果策略更新时参考也变，就不是这里固定参考的标准目标。教学实验保留独立参考副本，以便审计；这会有额外权重内存，而不是所谓“LoRA后只剩少量参数”。

In [8]:
print("同一个对象?", probe_policy is probe_reference)
print("参考中可训练参数:", sum(p.numel() for p in probe_reference.parameters() if p.requires_grad))
print("参考模式training:", probe_reference.training)
print("两者基座内容一致:", tensor_hash(base_state(probe_policy)) == tensor_hash(base_state(probe_reference)))
print("参考与策略是否存在共享参数存储:",
      bool({p.data_ptr() for p in probe_reference.parameters()} &
           {p.data_ptr() for p in probe_policy.parameters()}))

同一个对象? False
参考中可训练参数: 0
参考模式training: False
两者基座内容一致: True
参考与策略是否存在共享参数存储: False


## 5. 从 `[B,T,V]` 得到一条回答的 log 概率

自回归分解：

\[
\log\pi_\theta(y\mid x)
=\sum_{t=1}^{|y|}\log\pi_\theta(y_t\mid x,y_{<t})
\]

本课的 y 包括 EOS。使用 `log_softmax`，按目标编号 `gather`，屏蔽提示与 padding，沿时间轴求**和**。

```python
log_probs = F.log_softmax(logits[:, :-1], dim=-1)
targets = labels[:, 1:]
mask = targets != -100
safe_targets = targets.masked_fill(~mask, 0)
token_logps = log_probs.gather(-1, safe_targets[..., None]).squeeze(-1)
sequence_logps = token_logps.masked_fill(~mask, 0).sum(-1)
```

`-100` 不能直接交给 gather 当词表编号；先替换成安全下标，再把对应值忽略。`log_softmax` 避免先 softmax 再 log 的不稳定计算。[官方说明](https://docs.pytorch.org/docs/2.10/generated/torch.nn.functional.log_softmax.html)

**序列求和改成均值会改变目标。** 短回答 logp 为 -0.2，长回答两个 token 各 -0.15：求和偏好短回答（-0.2 > -0.3），平均却偏好长回答（-0.2 < -0.15）。标准序列 DPO 不默认替你做长度平均。

In [9]:
def sequence_logps(logits: torch.Tensor, labels: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """返回每条回答的log概率和及有效token数；不含prompt/PAD，包含EOS。"""
    require(logits.ndim == 3 and logits.is_floating_point(), "logits需要[B,T,V]")
    require(labels.shape == logits.shape[:2] and labels.dtype == torch.long, "标签形状或类型错误")
    require(logits.device == labels.device and logits.shape[1] >= 2, "设备或序列长度不合要求")
    require(bool(torch.isfinite(logits).all()), "logits不能有NaN/inf")
    require(bool((labels[:, 0] == IGNORE).all()), "首位置无前驱预测，本课必须忽略")
    shifted_labels = labels[:, 1:]
    mask = shifted_labels.ne(IGNORE)
    counts = mask.sum(dim=-1)
    require(bool((counts > 0).all()), "每个completion必须有有效目标")
    require(bool(((shifted_labels[mask] >= 0) &
                  (shifted_labels[mask] < logits.shape[-1])).all()), "标签编号越界")
    safe_labels = shifted_labels.masked_fill(~mask, 0)
    # 半精度下提升计算；FP64机制测试保留FP64。
    x = logits[:, :-1, :]
    if x.dtype in (torch.float16, torch.bfloat16):
        x = x.float()
    token_logps = F.log_softmax(x, dim=-1).gather(-1, safe_labels.unsqueeze(-1)).squeeze(-1)
    sums = token_logps.masked_fill(~mask, 0).sum(dim=-1)
    return sums, counts


def dpo_losses(pc: torch.Tensor, pr: torch.Tensor, rc: torch.Tensor,
               rr: torch.Tensor, beta: float = 0.1) -> torch.Tensor:
    """标准sigmoid-DPO，一对一个loss；reference明确不参与梯度。"""
    require(type(beta) in (int, float) and math.isfinite(beta) and beta > 0, "beta必须有限为正")
    tensors = [pc, pr, rc, rr]
    for t in tensors:
        require(isinstance(t, torch.Tensor) and t.ndim == 1 and t.numel() > 0,
                "四组logp应为非空向量")
        require(t.is_floating_point() and bool(torch.isfinite(t).all()), "logp必须为有限浮点")
        require(t.shape == pc.shape and t.device == pc.device, "logp未按偏好对对齐")
    margin = (pc - pr) - (rc.detach() - rr.detach())
    return F.softplus(-float(beta) * margin)


def pair_logps(model: TinyLM, batch: dict) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    values, counts = sequence_logps(model(batch["input_ids"]), batch["labels"])
    b = batch["pair_count"]
    require(values.numel() == 2 * b, "成对batch顺序或数量错误")
    return values[:b], values[b:], counts


def audit_training_objects(policy: TinyLM, reference: TinyLM,
                           optimizer: torch.optim.Optimizer) -> None:
    require(policy is not reference, "策略和参考不能是同一对象")
    require(all(not p.requires_grad and p.grad is None for p in reference.parameters()),
            "参考模型必须冻结且没有残留梯度")
    require(not reference.training, "参考模型应处于eval模式")
    require(all(p.requires_grad == is_adapter(n) for n, p in policy.named_parameters()),
            "只能训练策略adapter")
    storage = {p.data_ptr() for p in policy.parameters()}
    require(all(p.data_ptr() not in storage for p in reference.parameters()), "参考与策略共享存储")
    opt = [p for group in optimizer.param_groups for p in group["params"]]
    trainable = [p for p in policy.parameters() if p.requires_grad]
    require(len(opt) == len({id(p) for p in opt}), "优化器参数重复")
    require({id(p) for p in opt} == {id(p) for p in trainable}, "优化器没有精确指向adapter")

In [10]:
# 两条回答，长度不同。全零logits对应每个token概率均匀。
labels_demo = torch.tensor([
    [IGNORE, IGNORE, 3, EOS, IGNORE],
    [IGNORE, IGNORE, 3, 4, EOS],
])
logits_demo = torch.zeros(2, 5, 5, dtype=torch.float64, requires_grad=True)
logp_demo, count_demo = sequence_logps(logits_demo, labels_demo)
print("有效目标数:", count_demo.tolist())
print("回答logp和:", logp_demo.tolist())
print("手算:", (-count_demo.double()*math.log(5)).tolist())
(-logp_demo.sum()).backward()
print("首个预测位置的直接梯度（目标被忽略）:", logits_demo.grad[:,0,:])
print("最后预测位置的梯度（没有后继目标）:", logits_demo.grad[:,-1,:])

有效目标数: [2, 3]
回答logp和: [-3.2188758248682006, -4.828313737302301]
手算: [-3.2188758248682006, -4.828313737302301]
首个预测位置的直接梯度（目标被忽略）: tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]], dtype=torch.float64)
最后预测位置的梯度（没有后继目标）: tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]], dtype=torch.float64)


## 6. 标准 sigmoid-DPO：四组概率的差值

每个偏好对计算：

\[
\delta_i=(\ell^+_{\theta,i}-\ell^-_{\theta,i})
 -(\ell^+_{\mathrm{ref},i}-\ell^-_{\mathrm{ref},i})
\]

\[
L_i=-\log\sigma(\beta\delta_i)
=\operatorname{softplus}(-\beta\delta_i)
\]

最终对**偏好对**取均值。这里没有另训奖励模型，也不在每次 DPO 更新中在线采样回答。[DPO 原论文](https://arxiv.org/abs/2305.18290)

`beta` 不是学习率或解码温度。它出现在参考约束推导与损失缩放中；不能仅凭训练loss大小跨不同beta比较效果，也不能简单套用“越大一定更新越快”。

本课固定一种 sigmoid-DPO，不混入长度归一化、SFT混合项、标签平滑或其他偏好优化变体。

In [11]:
pc = torch.tensor([-2.0, -5.0], dtype=torch.float64, requires_grad=True)
pr = torch.tensor([-3.0, -4.0], dtype=torch.float64, requires_grad=True)
rc = pc.detach().clone().requires_grad_()
rr = pr.detach().clone().requires_grad_()
losses = dpo_losses(pc, pr, rc, rr, beta=0.1)
losses.mean().backward()
print("每对loss:", losses.detach().tolist(), "log2=", math.log(2))
print("策略chosen梯度:", pc.grad.tolist())
print("策略rejected梯度:", pr.grad.tolist())
print("参考梯度:", rc.grad, rr.grad)

每对loss: [0.6931471805599453, 0.6931471805599453] log2= 0.6931471805599453
策略chosen梯度: [-0.025, -0.025]
策略rejected梯度: [0.025, 0.025]
参考梯度: None None


### 6.1 初始 loss 为 log 2，既是检查点，也可能是假象

初始策略等于参考，则所有 \(\delta_i=0\)，所以 loss≈0.693147。

**这不意味着策略对两份回答各给50%概率。** 即使它原来更偏好 rejected，减去相同参考差值后 margin 仍为零。

错误实现也可能一直返回 log 2：如果直接把同一计算图当参考，`(pc-pr)-(pc-pr)` 恒为零，梯度也抵消。

把当前策略的值每步 detach 后当参考，虽然不会以同样方式抵消梯度，但锚点仍在移动，也不等同于固定参考 DPO。应检查引用关系、内容哈希与梯度，而不只看loss数值。

In [12]:
def mechanisms() -> dict:
    pc = torch.tensor([-2.0], dtype=torch.float64, requires_grad=True)
    pr = torch.tensor([-3.0], dtype=torch.float64, requires_grad=True)
    rc = pc.detach().clone().requires_grad_()
    rr = pr.detach().clone().requires_grad_()
    loss = dpo_losses(pc, pr, rc, rr, beta=0.1).mean()
    loss.backward()
    # 错误参考选成当前policy并且不detach：差值恒等于0且梯度为0。
    a = torch.tensor([-2.0], dtype=torch.float64, requires_grad=True)
    b = torch.tensor([-3.0], dtype=torch.float64, requires_grad=True)
    bad_loss = F.softplus(-0.1 * ((a - b) - (a - b))).mean()
    bad_loss.backward()
    # 同一个数值loss=log2，不保证策略本来偏好chosen。
    ref_p = torch.tensor([0.1, 0.8, 0.1], dtype=torch.float64)
    other_p = torch.tensor([0.4, 0.4, 0.2], dtype=torch.float64)
    moved_p = torch.tensor([0.3, 0.1, 0.6], dtype=torch.float64)
    before = dpo_losses(other_p[:1].log(), other_p[1:2].log(),
                        other_p[:1].log(), other_p[1:2].log(), 0.5).item()
    after = dpo_losses(moved_p[:1].log(), moved_p[1:2].log(),
                       other_p[:1].log(), other_p[1:2].log(), 0.5).item()
    return {
        "initial_loss": loss.item(), "log2": math.log(2),
        "initial_score_grads": {"chosen": pc.grad.item(), "rejected": pr.grad.item(),
                                "reference_chosen": rc.grad, "reference_rejected": rr.grad},
        "shared_graph_bug": {"loss": bad_loss.item(), "chosen_gradient": a.grad.item(),
                             "rejected_gradient": b.grad.item()},
        "initial_wrong_preference": {"chosen_probability": ref_p[0].item(),
            "rejected_probability": ref_p[1].item(),
            "dpo_loss": dpo_losses(ref_p[:1].log(), ref_p[1:2].log(),
                                    ref_p[:1].log(), ref_p[1:2].log()).item()},
        "chosen_probability_can_drop": {"before": 0.4, "after": 0.3,
                                         "loss_before": before, "loss_after": after},
        "sequence_length_counterexample": {"short_logps": [-0.2],
            "long_logps": [-0.15, -0.15], "sum_winner": "short", "mean_winner": "long"},
        "important": "相对隐式奖励差值不是任务正确率；平均log-ratio不是完整KL散度。",
    }

In [13]:
mechanism_report = mechanisms()
print(json.dumps(mechanism_report, ensure_ascii=False, indent=2))

{
  "initial_loss": 0.6931471805599453,
  "log2": 0.6931471805599453,
  "initial_score_grads": {
    "chosen": -0.05,
    "rejected": 0.05,
    "reference_chosen": null,
    "reference_rejected": null
  },
  "shared_graph_bug": {
    "loss": 0.6931471805599453,
    "chosen_gradient": 0.0,
    "rejected_gradient": 0.0
  },
  "initial_wrong_preference": {
    "chosen_probability": 0.1,
    "rejected_probability": 0.8,
    "dpo_loss": 0.6931471805599453
  },
  "chosen_probability_can_drop": {
    "before": 0.4,
    "after": 0.3,
    "loss_before": 0.6931471805599453,
    "loss_after": 0.4557463944083262
  },
  "sequence_length_counterexample": {
    "short_logps": [
      -0.2
    ],
    "long_logps": [
      -0.15,
      -0.15
    ],
    "sum_winner": "short",
    "mean_winner": "long"
  },
  "important": "相对隐式奖励差值不是任务正确率；平均log-ratio不是完整KL散度。"
}


## 7. 参考概率可以缓存，但不能忘记它们属于谁

固定数据、固定参考模型、确定的模板与掩码、关闭随机Dropout后，可以提前计算参考 logp，避免每一步都重新跑参考前向。TRL 也提供相关预计算选项。[TRL DPO](https://huggingface.co/docs/trl/dpo_trainer)

缓存至少需要绑定：**参考权重、模型配置、词表、提示与两份回答、模板/掩码/移位/求和规则、精度与必要的执行条件**。不能只用题目ID匹配一个旧数字。

本课只预计算训练侧（可选），开发验证和最终测试仍在线计算。按 ID 和完整记录摘要查缓存，不按“当前第几行”匹配。

### 已有 SFT adapter 时的陷阱

如果生产基座的 SFT 能力仍在一个 adapter 中，直接关闭**全部**adapter会退回 SFT 之前的模型，而不是正确参考。要冻结对应 SFT checkpoint，或显式管理参考与训练 adapter。不要把“关闭adapter”等同于“永远拿到正确参考”。

本课 SFT 是完整权重更新，保存到了冻结基座中，所以只关闭新加的 DPO adapter，才恰好恢复本课参考。

In [14]:
def cache_contract(reference: TinyLM, codec: CharCodec) -> dict:
    return {"version": VERSION, "reference_hash": tensor_hash(base_state(reference)),
            "model_config": asdict(reference.cfg), "codec": codec.contract(),
            "mask_rule": "same-position/completion-plus-eos/right-pad/shift-once/sum-v1",
            "precision": str(next(reference.parameters()).dtype),
            "device_type": next(reference.parameters()).device.type}


def precompute_reference(reference: TinyLM, rows: list[dict], codec: CharCodec,
                         batch_size: int = 16) -> dict:
    require(not reference.training and all(not p.requires_grad for p in reference.parameters()),
            "只缓存冻结eval参考模型")
    require(type(batch_size) is int and batch_size > 0, "batch_size必须为正整数")
    require(len({r['id'] for r in rows}) == len(rows), "缓存ID重复")
    values = {}
    device = next(reference.parameters()).device
    with torch.no_grad():
        for i in range(0, len(rows), batch_size):
            subset = rows[i:i + batch_size]
            batch = collate_pairs(subset, codec, reference.cfg.max_len, device)
            c, r, _ = pair_logps(reference, batch)
            for j, row in enumerate(subset):
                values[row["id"]] = {"pair_digest": digest(row),
                                     "chosen": c[j].item(), "rejected": r[j].item()}
    return {"contract": cache_contract(reference, codec), "entries": values}


def cached_pair_logps(cache: dict, rows: list[dict], expected_contract: dict,
                      device: torch.device, dtype: torch.dtype) -> tuple[torch.Tensor, torch.Tensor]:
    require(cache.get("contract") == expected_contract, "参考缓存身份不匹配，必须重算")
    chosen, rejected = [], []
    for row in rows:
        entry = cache["entries"].get(row["id"])
        require(entry is not None and entry.get("pair_digest") == digest(row),
                "缓存缺题或提示/答案/顺序相关内容发生变化")
        for field in ["chosen", "rejected"]:
            require(type(entry[field]) in (int, float) and math.isfinite(entry[field]), "缓存数值非法")
        chosen.append(entry["chosen"]); rejected.append(entry["rejected"])
    return (torch.tensor(chosen, device=device, dtype=dtype),
            torch.tensor(rejected, device=device, dtype=dtype))

In [15]:
cached_demo = precompute_reference(probe_reference, demo_rows, codec, batch_size=2)
expected_contract = cache_contract(probe_reference, codec)
cache_c, cache_r = cached_pair_logps(cached_demo, list(reversed(demo_rows)),
                                    expected_contract, torch.device("cpu"), torch.float32)
print("以反向题目顺序查询缓存:", cache_c.tolist(), cache_r.tolist())
modified_rows = copy.deepcopy(demo_rows)
modified_rows[0]["chosen"] = "3"
try:
    cached_pair_logps(cached_demo, modified_rows, expected_contract,
                      torch.device("cpu"), torch.float32)
except LessonError as error:
    print("预期拒绝:", error)

以反向题目顺序查询缓存: [-6.161160469055176, -9.253622055053711] [-9.37762451171875, -9.23030948638916]
预期拒绝: 缓存缺题或提示/答案/顺序相关内容发生变化


## 8. 一次 DPO 更新的完整顺序

```text
清理旧梯度
    ↓
策略：2B条序列前向 → 得到chosen/rejected logp
    ↓
参考：no_grad前向（或身份已核对的缓存）
    ↓
逐对DPO loss → pair均值 → backward
    ↓
仅裁剪adapter梯度 → optimizer.step
```

参考不是优化器的一部分。冻结基座的权重不接收梯度，但其计算仍可把梯度传给更早位置的可训练adapter，不要将整个策略前向包进 `no_grad()`。

**与SFT的token平均不同，本课DPO每个偏好对等权。** 将来做梯度累积时，用窗口总偏好对数归一化各小批次的loss和，不要机械照搬“按回答token数归一化”的SFT循环。

训练更新中报告的batch loss在不断变化的参数上得到；每轮dev则固定参数重新评估，两者也不宜误作完全相同的统计对象。

In [16]:
def sft_warmup(model: TinyLM, rows: list[dict], codec: CharCodec, cfg: TrainConfig) -> list[dict]:
    require(all(r["split"] == "train" for r in rows), "SFT只接收训练侧")
    require(all(p.requires_grad for p in model.parameters()), "SFT热身使用未冻结普通模型")
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.sft_lr, weight_decay=0.0)
    rng = random.Random(cfg.seed + 1)
    device = next(model.parameters()).device
    history = []
    for step in range(1, cfg.sft_steps + 1):
        sample = rng.sample(rows, min(cfg.batch_size, len(rows)))
        batch = collate_pairs(sample, codec, model.cfg.max_len, device)
        b = len(sample)
        optimizer.zero_grad(set_to_none=True)
        logps, counts = sequence_logps(model(batch["input_ids"][:b]), batch["labels"][:b])
        loss = -logps.sum() / counts.sum()
        require(bool(torch.isfinite(loss)), "SFT loss非有限")
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm, error_if_nonfinite=True)
        optimizer.step()
        if step == 1 or step % 20 == 0 or step == cfg.sft_steps:
            history.append({"step": step, "sample_token_nll": loss.item()})
    optimizer.zero_grad(set_to_none=True)
    model.eval()
    return history


def one_dpo_step(policy: TinyLM, reference: TinyLM, batch: dict,
                 optimizer: torch.optim.Optimizer, beta: float, max_grad_norm: float = 1.0,
                 ref_values: tuple[torch.Tensor, torch.Tensor] | None = None) -> dict:
    optimizer.zero_grad(set_to_none=True)
    policy.train()  # 所有Dropout仍为0；train()不意味着冻结基座会被更新。
    pc, pr, _ = pair_logps(policy, batch)
    if ref_values is None:
        with torch.no_grad():
            rc, rr, _ = pair_logps(reference, batch)
    else:
        rc, rr = ref_values
    losses = dpo_losses(pc, pr, rc, rr, beta)
    loss = losses.mean()  # 本课每个偏好对等权；不是按回答token数加权。
    loss.backward()
    adapter_parameters = [p for p in policy.parameters() if p.requires_grad]
    require(all(p.grad is not None and bool(torch.isfinite(p.grad).all())
                for p in adapter_parameters), "adapter梯度缺失或非有限")
    norm = nn.utils.clip_grad_norm_(adapter_parameters, max_grad_norm, error_if_nonfinite=True)
    optimizer.step()
    return {"loss": loss.item(), "preclip_grad_norm": float(norm),
            "pair_count": batch["pair_count"]}


@torch.no_grad()
def evaluate_pairs(policy: TinyLM, reference: TinyLM, rows: list[dict], codec: CharCodec,
                   beta: float, batch_size: int = 16) -> dict:
    require(bool(rows), "评测集合不能为空")
    mode = policy.training
    policy.eval(); reference.eval()
    device = next(policy.parameters()).device
    total_loss = total_delta = total_gap = cnll = rnll = 0.0
    ccount = rcount = wins = relative_wins = 0
    records = []
    try:
        for i in range(0, len(rows), batch_size):
            subset = rows[i:i + batch_size]
            batch = collate_pairs(subset, codec, policy.cfg.max_len, device)
            pc, pr, counts = pair_logps(policy, batch)
            rc, rr, _ = pair_logps(reference, batch)
            losses = dpo_losses(pc, pr, rc, rr, beta)
            raw_gap = pc - pr
            relative_gap = raw_gap - (rc - rr)
            total_loss += losses.sum().item()
            total_gap += raw_gap.sum().item()
            total_delta += relative_gap.sum().item()
            wins += int((raw_gap > 0).sum())
            relative_wins += int((relative_gap > 1e-7).sum())
            cnll += (-pc.sum()).item(); rnll += (-pr.sum()).item()
            ccount += int(counts[:len(subset)].sum()); rcount += int(counts[len(subset):].sum())
            for j, row in enumerate(subset):
                records.append({"id": row["id"], "policy_chosen_logp": pc[j].item(),
                    "policy_rejected_logp": pr[j].item(), "reference_chosen_logp": rc[j].item(),
                    "reference_rejected_logp": rr[j].item(), "raw_pair_margin": raw_gap[j].item(),
                    "reference_corrected_margin": relative_gap[j].item(), "dpo_loss": losses[j].item()})
    finally:
        policy.train(mode)
    n = len(rows)
    return {"pairs": n, "dpo_loss": total_loss / n,
            "raw_pair_accuracy": wins / n, "relative_margin_positive_rate": relative_wins / n,
            "mean_raw_margin": total_gap / n, "mean_reference_corrected_margin": total_delta / n,
            "chosen_token_nll": cnll / ccount, "rejected_token_nll": rnll / rcount,
            "records": records}


@torch.no_grad()
def greedy_generate(model: TinyLM, codec: CharCodec, prompt: str, max_new_tokens: int = 6) -> dict:
    require(type(max_new_tokens) is int and max_new_tokens > 0, "输出预算须为正整数")
    ids = [BOS] + codec.encode(prompt)
    require(0 < len(ids) < model.cfg.max_len, "提示没有生成空间")
    mode = model.training
    model.eval()
    device = next(model.parameters()).device
    out = []
    stop = "max_new_tokens"
    try:
        for _ in range(max_new_tokens):
            if len(ids) >= model.cfg.max_len:
                stop = "context_limit"; break
            logits = model(torch.tensor([ids], dtype=torch.long, device=device))[0, -1].clone()
            logits[[PAD, BOS]] = -torch.inf
            token = int(logits.argmax())
            ids.append(token)
            out.append(token)
            if token == EOS:
                stop = "eos"; break
    finally:
        model.train(mode)
    text_ids = out[:-1] if out and out[-1] == EOS else out
    return {"text": codec.decode(text_ids), "new_ids": out, "stop_reason": stop}


def generation_evaluation(model: TinyLM, rows: list[dict], codec: CharCodec) -> dict:
    require(bool(rows), "生成评测集合为空")
    records = []
    for row in rows:
        answer = greedy_generate(model, codec, row["prompt"])
        records.append({"id": row["id"], "prompt": row["prompt"], "expected": row["expected"],
                        **answer, "correct": answer["text"] == row["expected"] and
                                              answer["stop_reason"] == "eos"})
    correct = sum(r["correct"] for r in records)
    return {"planned": len(rows), "correct": correct,
            "strict_accuracy": correct / len(rows), "records": records}

In [17]:
probe_optimizer = torch.optim.AdamW(
    [p for p in probe_policy.parameters() if p.requires_grad], lr=0.001, weight_decay=0.0
)
audit_training_objects(probe_policy, probe_reference, probe_optimizer)
frozen_before = tensor_hash(base_state(probe_policy))
step_result = one_dpo_step(probe_policy, probe_reference, demo_batch, probe_optimizer, beta=0.1)
print("一步结果:", step_result)
for name, parameter in probe_policy.named_parameters():
    if is_adapter(name):
        print(name, "grad_norm=", parameter.grad.norm().item())
print("基座未变:", tensor_hash(base_state(probe_policy)) == frozen_before)
print("参考grad全部为空:", all(p.grad is None for p in probe_reference.parameters()))

一步结果: {'loss': 0.6931471824645996, 'preclip_grad_norm': 0.035423725843429565, 'pair_count': 2}
blocks.0.attn.q_proj.lora_A grad_norm= 0.0
blocks.0.attn.q_proj.lora_B grad_norm= 0.00073876918759197
blocks.0.attn.v_proj.lora_A grad_norm= 0.0
blocks.0.attn.v_proj.lora_B grad_norm= 0.02783842757344246
blocks.1.attn.q_proj.lora_A grad_norm= 0.0
blocks.1.attn.q_proj.lora_B grad_norm= 0.0006655247998423874
blocks.1.attn.v_proj.lora_A grad_norm= 0.0
blocks.1.attn.v_proj.lora_B grad_norm= 0.021883180364966393
基座未变: True
参考grad全部为空: True


### 8.1 A首步梯度为零，不要误判为训练没有连上

本课初始化 A 随机、B 全零，增量 \(BA=0\)。首步 A 的梯度为零，B 可以收到非零梯度；更新B以后，A才开始获得非零梯度。

上面是随机小模型的一步连接实验，不是完整训练的参考状态。完整实验会重新设置固定种子、先完成SFT，再建立独立参考与策略。

## 9. 完整实验协议：先确定选择规则，再打开测试集

默认配置：

| 阶段 | 设置 |
|---|---|
| SFT起点 | 训练侧chosen，160次参数更新，batch=16 |
| DPO | 8轮，每轮10个偏好batch，80次更新 |
| LoRA | 每层Q/V、r=4、alpha=8，共1536个可训练参数 |
| DPO参数 | beta=0.1，AdamW lr=0.001，weight_decay=0 |
| 模型选择 | dev的pair平均DPO loss最低，包含epoch 0，并列取更早轮 |
| 最终生成 | 每题只给提示，贪心，最多6个新token，EOS结束 |

这不是为了证明DPO必然提升。**不因为看到测试退步就重新改beta、rank、轮数。** 研究这些变化时要把当前测试集视为已见回归材料，另留新的最终评测。

源码中的规则答案都是公开的，不能称为保密能力基准。

In [18]:
def run_experiment(output_root: Path = Path("lesson_58_outputs"),
                   config: TrainConfig | None = None, device_name: str = "cpu",
                   verbose: bool = True) -> Path:
    cfg = config or TrainConfig()
    cfg.validate()
    device = resolve_device(device_name)
    seed_all(cfg.seed)
    torch.set_num_threads(1)
    codec = CharCodec()
    splits = make_splits(cfg.seed)
    validate_splits(splits)
    mc = ModelConfig(vocab_size=len(codec.tokens))
    root = Path(output_root)
    root.mkdir(parents=True, exist_ok=True)
    out = root / ("run_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f"))
    out.mkdir(exist_ok=False)
    manifest = {"version": VERSION, "status": "running", "python": platform.python_version(),
                "torch": str(torch.__version__), "device": str(device), "dtype": "torch.float32",
                "sources": SOURCES, "config": asdict(cfg),
                "scope": "离线合成任务；真实小Transformer训练；非预训练大模型/PEFT/QLoRA"}
    write_json(out / "manifest.json", manifest)
    try:
        write_json(out / "data.json", splits)
        write_json(out / "mechanisms.json", mechanisms())
        base = TinyLM(mc).to(device)
        sft_history = sft_warmup(base, splits["train"], codec, cfg)
        reference = freeze_reference(base)
        reference_hash = tensor_hash(base_state(reference))
        torch.save({"format": VERSION, "config": asdict(mc), "codec": codec.contract(),
                    "base_hash": reference_hash,
                    "state_dict": {n: t.detach().cpu() for n, t in reference.state_dict().items()}},
                   out / "sft_reference.pt")
        seed_all(cfg.seed + 100)
        policy = inject_lora(copy.deepcopy(base), cfg.rank, cfg.alpha)
        del base
        optimizer = torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad],
                                      lr=cfg.dpo_lr, weight_decay=0.0)
        audit_training_objects(policy, reference, optimizer)
        require(tensor_hash(base_state(policy)) == reference_hash, "初始基座与参考不匹配")
        initial_dev = evaluate_pairs(policy, reference, splits["dev"], codec, cfg.beta)
        require(abs(initial_dev["dpo_loss"] - math.log(2)) < 1e-6, "初始DPO loss应为log2")
        cache = precompute_reference(reference, splits["train"], codec, cfg.batch_size) if cfg.cached_reference else None
        expected_cache = cache_contract(reference, codec)
        if cache is not None:
            write_json(out / "reference_cache_train.json", cache)
        history = [{"epoch": 0, "dev": {k: v for k, v in initial_dev.items() if k != "records"}}]
        best_loss, best_epoch, best_state = initial_dev["dpo_loss"], 0, adapter_snapshot(policy)
        updates, first_grad = 0, None
        for epoch in range(1, cfg.dpo_epochs + 1):
            order = list(splits["train"])
            random.Random(cfg.seed + 1000 + epoch).shuffle(order)
            train_loss_sum = 0.0
            for i in range(0, len(order), cfg.batch_size):
                subset = order[i:i + cfg.batch_size]
                batch = collate_pairs(subset, codec, mc.max_len, device)
                ref_values = None if cache is None else cached_pair_logps(
                    cache, subset, expected_cache, device, next(policy.parameters()).dtype)
                result = one_dpo_step(policy, reference, batch, optimizer, cfg.beta,
                                      cfg.max_grad_norm, ref_values)
                updates += 1
                train_loss_sum += result["loss"] * len(subset)
                if first_grad is None:
                    first_grad = {name: p.grad.norm().item() for name, p in policy.named_parameters()
                                  if is_adapter(name)}
            # 每轮验证身份，预计算模式也不能忽略参考/基座变化。
            require(tensor_hash(base_state(reference)) == reference_hash, "参考模型被更新")
            require(tensor_hash(base_state(policy)) == reference_hash, "策略冻结基座被更新")
            audit_training_objects(policy, reference, optimizer)
            dev = evaluate_pairs(policy, reference, splits["dev"], codec, cfg.beta)
            history.append({"epoch": epoch, "train_step_dpo_loss_mean": train_loss_sum / len(order),
                            "dev": {k: v for k, v in dev.items() if k != "records"}})
            if dev["dpo_loss"] < best_loss:  # 严格改善；并列保留更早轮次。
                best_loss, best_epoch, best_state = dev["dpo_loss"], epoch, adapter_snapshot(policy)
            if verbose:
                print(f"DPO epoch={epoch:2d} dev_loss={dev['dpo_loss']:.6f} raw_pair_acc={dev['raw_pair_accuracy']:.3f}")
        last_dev = history[-1]["dev"]
        load_adapter(policy, best_state)
        policy.eval()
        # 先锁定候选并落盘，再读取test进行最终评估。
        selection = {"criterion": "dev_pair_mean_dpo_loss", "selected_epoch": best_epoch,
                     "selected_dev_loss": best_loss, "includes_epoch_zero": True,
                     "test_used_for_selection": False,
                     "adapter_hash": tensor_hash(best_state), "reference_hash": reference_hash}
        write_json(out / "selection.json", selection)
        adapter_bundle = {"format": VERSION, "config": asdict(mc), "codec": codec.contract(),
                          "rank": cfg.rank, "alpha": cfg.alpha, "beta": cfg.beta,
                          "reference_hash": reference_hash, "adapter": best_state,
                          "selection": selection}
        torch.save(adapter_bundle, out / "dpo_adapter.pt")
        before_pairs = evaluate_pairs(reference, reference, splits["test"], codec, cfg.beta)
        after_pairs = evaluate_pairs(policy, reference, splits["test"], codec, cfg.beta)
        before_gen = generation_evaluation(reference, splits["test"], codec)
        after_gen = generation_evaluation(policy, splits["test"], codec)
        baseline_by = {r["id"]: r for r in before_gen["records"]}
        improved = [r["id"] for r in after_gen["records"] if r["correct"] and not baseline_by[r["id"]]["correct"]]
        regressed = [r["id"] for r in after_gen["records"] if not r["correct"] and baseline_by[r["id"]]["correct"]]
        probe_rows = splits["dev"][:3]
        probe_batch = collate_pairs(probe_rows, codec, mc.max_len, device)
        with torch.no_grad(), adapter_disabled(policy):
            disabled_diff = (policy(probe_batch["input_ids"]) - reference(probe_batch["input_ids"])).abs().max().item()
        require(disabled_diff == 0.0, "关闭新增DPO adapter应恢复冻结SFT基座")
        with torch.no_grad():
            probe_logits = policy(probe_batch["input_ids"]).cpu()
        probe_generations = [greedy_generate(policy, codec, r["prompt"]) for r in probe_rows]
        torch.save({"input_ids": probe_batch["input_ids"].cpu(), "logits": probe_logits,
                    "prompts": [r["prompt"] for r in probe_rows], "generations": probe_generations},
                   out / "reload_probe.pt")
        metrics = {"model_parameters": sum(p.numel() for p in reference.parameters()),
                   "policy_total_parameters": sum(p.numel() for p in policy.parameters()),
                   "trainable_lora_parameters": sum(p.numel() for p in policy.parameters() if p.requires_grad),
                   "split_sizes": {k: len(v) for k, v in splits.items()}, "updates": updates,
                   "selection": selection, "last_dev": last_dev,
                   "test_before": {k: v for k, v in before_pairs.items() if k != "records"},
                   "test_after": {k: v for k, v in after_pairs.items() if k != "records"},
                   "generation_before": {k: v for k, v in before_gen.items() if k != "records"},
                   "generation_after": {k: v for k, v in after_gen.items() if k != "records"},
                   "improved_ids": improved, "regressed_ids": regressed,
                   "reference_unchanged": tensor_hash(base_state(reference)) == reference_hash,
                   "base_unchanged": tensor_hash(base_state(policy)) == reference_hash,
                   "disabled_adapter_logit_difference": disabled_diff,
                   "first_step_adapter_gradient_norms": first_grad,
                   "limitations": ["单种子合成复制任务，三侧共享语法与取值范围。",
                                    "偏好标签由规则构造，不是人类标注或真实采样。",
                                    "开发集DPO loss选轮，不保证自由生成或旧任务改善。",
                                    "未实现量化、混合精度、KV cache、TRL/PEFT或完整续训。"]}
        write_json(out / "training.json", {"sft": sft_history, "dpo": history})
        write_json(out / "metrics.json", metrics)
        write_json(out / "pair_predictions.json", {"before": before_pairs["records"], "after": after_pairs["records"]})
        write_json(out / "generations.json", {"before": before_gen["records"], "after": after_gen["records"]})
        reload_report = reload_run(out, device_name=str(device))
        write_json(out / "reload.json", reload_report)
        manifest["status"] = "completed"
        write_json(out / "manifest.json", manifest)
        if verbose:
            print(json.dumps(metrics, ensure_ascii=False, indent=2))
            print("报告目录：", out.resolve())
        return out
    except Exception as exc:
        manifest.update(status="failed", error_type=type(exc).__name__, error=str(exc))
        write_json(out / "manifest.json", manifest)
        raise


def reload_run(directory: Path, device_name: str = "cpu") -> dict:
    """只读取可信的本课文件，weights_only限制反序列化，不是安全来源证明。"""
    directory = Path(directory)
    device = resolve_device(device_name)
    base = torch.load(directory / "sft_reference.pt", map_location="cpu", weights_only=True)
    adapter = torch.load(directory / "dpo_adapter.pt", map_location="cpu", weights_only=True)
    probe = torch.load(directory / "reload_probe.pt", map_location="cpu", weights_only=True)
    require(base["format"] == adapter["format"] == VERSION, "文件格式不匹配")
    codec = CharCodec()
    require(base["codec"] == adapter["codec"] == codec.contract(), "词表或输入约定不匹配")
    require(base["config"] == adapter["config"], "模型结构不匹配")
    require(tensor_hash(base["state_dict"]) == base["base_hash"] == adapter["reference_hash"],
            "adapter没有匹配到精确SFT参考基座")
    require(tensor_hash(adapter["adapter"]) == adapter["selection"]["adapter_hash"], "adapter内容变化")
    model = TinyLM(ModelConfig(**base["config"])).to(device)
    model.load_state_dict(base["state_dict"], strict=True)
    inject_lora(model, adapter["rank"], adapter["alpha"])
    load_adapter(model, adapter["adapter"])
    model.eval().requires_grad_(False)
    with torch.no_grad():
        logits = model(probe["input_ids"].to(device)).cpu()
    delta = (logits - probe["logits"]).abs().max().item()
    require(torch.allclose(logits, probe["logits"], atol=2e-5, rtol=2e-5), "重载logits不一致")
    generated = [greedy_generate(model, codec, prompt) for prompt in probe["prompts"]]
    require(generated == probe["generations"], "重载生成不一致；检查精度、设备与权重")
    return {"format": VERSION, "device": str(device), "maximum_logit_difference": delta,
            "generated_ids_equal": True, "reference_identity_verified": True,
            "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
            "exact_resume": False}

In [19]:
RUN_TRAINING = True
RUN_DIR = None
if RUN_TRAINING:
    training_config = TrainConfig()  # 固定默认值；修改后结果应以新报告为准。
    RUN_DIR = run_experiment(OUTPUT_ROOT, training_config, DEVICE, verbose=False)
    print("训练报告目录:", RUN_DIR)
else:
    print("完整训练跳过。上面的数学与梯度实验仍然可用。")

训练报告目录: lesson_58_outputs/run_20261103_022031_549078


## 10. 三种指标，不能混为一个“DPO准确率”

1. **原始成对排序**：\(\log\pi_\theta(y^+|x)>\log\pi_\theta(y^-|x)\) 是否成立。
2. **参考校正margin**：当前相对偏好是否比参考更强；初始值是0，不能叫作原始正确率。
3. **自由生成严格成功**：只给提示，逐步接回自己生成的token，结果是否等于规定答案且正常EOS结束。

报告还分别统计chosen和rejected的token NLL。DPO可以通过更大幅度降低rejected的概率来改善相对margin，同时让chosen绝对概率下降。

**这里的平均log-ratio不是对所有可能回答积分得到的完整KL散度，不把它命名为“真实KL”。**

In [20]:
if RUN_DIR is not None:
    metrics = json.loads((RUN_DIR / "metrics.json").read_text(encoding="utf-8"))
    print("模型参数 / LoRA参数:", metrics["model_parameters"], metrics["trainable_lora_parameters"])
    print("所选轮次:", metrics["selection"]["selected_epoch"])
    for side in ["before", "after"]:
        pair = metrics[f"test_{side}"]
        gen = metrics[f"generation_{side}"]
        print(side, {"DPO loss":pair["dpo_loss"], "pair accuracy":pair["raw_pair_accuracy"],
                     "chosen token NLL":pair["chosen_token_nll"],
                     "rejected token NLL":pair["rejected_token_nll"],
                     "生成正确":f"{gen['correct']}/{gen['planned']}"})

模型参数 / LoRA参数: 41184 1536
所选轮次: 8
before {'DPO loss': 0.6931471824645996, 'pair accuracy': 1.0, 'chosen token NLL': 0.00779947087544353, 'rejected token NLL': 4.818384499385439, '生成正确': '48/48'}
after {'DPO loss': 0.5780532956123352, 'pair accuracy': 1.0, 'chosen token NLL': 0.207564149584089, 'rejected token NLL': 6.158928049021754, '生成正确': '45/48'}


### 10.1 制作环境实际出现了退步——保留，不修饰

默认CPU运行中，测试DPO loss从约 **0.693147** 降为 **0.578053**，给定两候选的排序仍是 **48/48**；但自由生成从 **48/48** 降为 **45/48**。

chosen 的token NLL由约 **0.007799** 上升到 **0.207564**，rejected由约 **4.818384** 上升到 **6.158928**。rejected的序列概率被压得更多，相对目标改善，但chosen本身也失去了一些概率。

以下单元展示三个实际退步：目标为负数，模型多生成了一个尾随负号。这种回答不在人工构造的错误候选中，但自由生成仍然可能产生。

这不是“DPO普遍有害”的结论，而是一个完整工程反例：**偏好目标收敛，不自动等于任务改善。** 这些数字只对应固定版本、种子与CPU环境；改动配置后以实际记录为准。

In [21]:
if RUN_DIR is not None:
    generations = json.loads((RUN_DIR / "generations.json").read_text(encoding="utf-8"))
    before = {r["id"]: r for r in generations["before"]}
    for record in generations["after"]:
        if before[record["id"]]["correct"] and not record["correct"]:
            print({"prompt":record["prompt"], "expected":record["expected"],
                   "SFT":before[record["id"]]["text"], "DPO":record["text"],
                   "stop":record["stop_reason"]})
    print("改善题:", metrics["improved_ids"])
    print("退步题:", metrics["regressed_ids"])

{'prompt': 'last:3,-3=', 'expected': '-3', 'SFT': '-3', 'DPO': '-3-', 'stop': 'eos'}
{'prompt': 'last:9,-2=', 'expected': '-2', 'SFT': '-2', 'DPO': '-2-', 'stop': 'eos'}
{'prompt': 'last:0,-2=', 'expected': '-2', 'SFT': '-2', 'DPO': '-2-', 'stop': 'eos'}
改善题: []
退步题: ['pair_3_-3', 'pair_9_-2', 'pair_0_-2']


### 10.2 本课能支持什么结论？

能验证：完整Transformer上的序列DPO可运行；只有adapter更新；参考与基座不变；指标和生成可以不同步。

不能推导：真实中文能力、推理能力、最佳beta/rank、LoRA优于全量训练、GPU速度或显存节省。

SFT已经解决了这份简单任务，不等于它解决了一般推理。DPO继续使用同一训练侧来源，也不构成与等算力SFT继续训练的公平算法排名。

下一轮改进可以在开发侧检查：chosen概率、结束符行为、偏好覆盖与生成任务约束；但不要据此把当前测试集继续包装为未知测试。

In [22]:
if RUN_DIR is not None:
    print("参考不变:", metrics["reference_unchanged"])
    print("策略冻结基座不变:", metrics["base_unchanged"])
    print("关闭新增adapter后，与SFT参考的最大logits差:", metrics["disabled_adapter_logit_difference"])
    print("首步adapter梯度:")
    print(json.dumps(metrics["first_step_adapter_gradient_norms"], indent=2))

参考不变: True
策略冻结基座不变: True
关闭新增adapter后，与SFT参考的最大logits差: 0.0
首步adapter梯度:
{
  "blocks.0.attn.q_proj.lora_A": 0.0,
  "blocks.0.attn.q_proj.lora_B": 0.02731253392994404,
  "blocks.0.attn.v_proj.lora_A": 0.0,
  "blocks.0.attn.v_proj.lora_B": 0.09527353942394257,
  "blocks.1.attn.q_proj.lora_A": 0.0,
  "blocks.1.attn.q_proj.lora_B": 0.0011468963930383325,
  "blocks.1.attn.v_proj.lora_A": 0.0,
  "blocks.1.attn.v_proj.lora_B": 0.015143013559281826
}


## 11. 保存与重载：adapter不能离开原SFT基座独立解释

完整运行写入新的时间戳目录，不覆盖以前实验：

| 文件 | 内容 |
|---|---|
| `sft_reference.pt` | SFT权重、配置、词表与身份摘要 |
| `dpo_adapter.pt` | Q/V的A/B、rank、alpha、beta、所选轮次与参考身份 |
| `selection.json` | 测试前就已固定的选择依据 |
| `training.json` | SFT采样loss和每轮DPO/dev指标 |
| `pair_predictions.json` | 逐题四组logp及相对margin |
| `generations.json` | 基座/adapter的全部原始生成 |
| `reload_probe.pt`、`reload.json` | 开发侧固定探针与重载比较 |

模型恢复会检查精确的参考权重、词表和配置，不只是形状能对上。只从可信来源加载，`weights_only=True` 不是来源认证。

没有保存完整优化器、随机状态和数据游标，**不是第26课的精确断点续训**。

In [23]:
if RUN_DIR is not None:
    print(json.dumps(reload_run(RUN_DIR, DEVICE), ensure_ascii=False, indent=2))

{
  "format": "lesson58-v1",
  "device": "cpu",
  "maximum_logit_difference": 0.0,
  "generated_ids_equal": true,
  "reference_identity_verified": true,
  "trainable_parameters": 0,
  "exact_resume": false
}


## 12. 单卡工程边界与运行方式

本课同时保留策略与参考两份小模型；LoRA只减少可训练参数，不消除基座或激活。

真实模型常用预计算参考logp、冻结副本、正确管理参考adapter等方式控制资源。需要确保SFT状态、模板、掩码、精度和权重都与缓存一致；不要把“关闭adapter”当作无条件等价方案。[TRL DPO](https://huggingface.co/docs/trl/dpo_trainer)

完整脚本：

```bash
python lesson_58_language_model_dpo_and_lora.py
python lesson_58_language_model_dpo_and_lora.py --cache-reference
python lesson_58_language_model_dpo_and_lora.py --math-only
python lesson_58_language_model_dpo_and_lora.py --reload-run lesson_58_outputs/你的实际目录
```

已有CUDA环境时可加 `--device cuda`；明确请求但不可用会报错。**制作环境没有CUDA，未在3060上实测，也没有性能或峰值显存基准。** 不同设备/版本不保证数值或生成逐位一致。

`--cache-reference`改变计算路径，不是另一种DPO目标。默认检查会比较缓存与在线参考的一步更新。

## 13. 自动检查：不要只测试loss会下降

检查覆盖：模板与padding、一次移位、序列和、偏好方向、参考冻结、参数隔离、零初始化、首次和再次梯度、缓存身份、批次对齐、保存与生成边界。

制作时还对临时脚本加入六种故障：标签错位、序列和改成均值、偏好方向反转、参考参与梯度、遗漏参考差值、初始LoRA增量非零。原有检查均检测到这些故障。

以下固定检查会真实运行小网络，但不重新运行完整160步SFT＋80步DPO。

In [24]:
def run_checks() -> dict:
    torch.set_num_threads(1)
    seed_all(5800)
    names = []
    def check(name, condition):
        if not bool(condition):
            raise AssertionError(name)
        names.append(name)
    def rejects(name, fn):
        try:
            fn()
        except (LessonError, ValueError):
            names.append(name); return
        raise AssertionError(name + "：未拒绝非法输入")
    codec = CharCodec()
    splits = make_splits()
    validate_splits(splits)
    check("固定划分160/48/48", [len(splits[s]) for s in ["train", "dev", "test"]] == [160,48,48])
    check("每题chosen符合明示规则", all(r["chosen"] == r["prompt"].split(",")[1][:-1]
                                           for rows in splits.values() for r in rows))
    rng_before = random.getstate()
    check("固定种子划分可复现", make_splits() == make_splits())
    check("局部打乱不修改全局随机状态", random.getstate() == rng_before)
    check("字符编码还原", codec.decode(codec.encode("last:-3,12=")) == "last:-3,12=")
    rejects("未知字符拒绝", lambda: codec.encode("新"))
    ids, labels = encode_completion(codec, "last:3,12=", "12", 32)
    prefix_len = 1 + len(codec.encode("last:3,12="))
    check("提示目标被屏蔽", labels[:prefix_len] == [IGNORE] * prefix_len)
    check("第一个回答标签同位", labels[prefix_len] == codec.to_id["1"])
    check("EOS参与监督", labels[-1] == EOS)
    rejects("超长不截断", lambda: encode_completion(codec, "last:3,12=", "12", 4))
    rejects("空答案拒绝", lambda: encode_completion(codec, "last:3,12=", "", 32))
    rows = [{"id":"x","prompt":"last:1,2=","chosen":"2","rejected":"12"},
            {"id":"y","prompt":"last:-3,12=","chosen":"12","rejected":"-3"}]
    batch = collate_pairs(rows, codec, 32)
    check("2B成对批次", batch["input_ids"].shape[0] == 4)
    for j, row in enumerate(rows):
        p = 1 + len(codec.encode(row["prompt"]))
        check(f"偏好对{j}前缀完全相同", torch.equal(batch["input_ids"][j,:p], batch["input_ids"][j+2,:p]))
    check("padding标签全忽略", bool((batch["labels"][~batch["attention_mask"]] == IGNORE).all()))
    rejects("空批次拒绝", lambda: collate_pairs([], codec, 32))
    rejects("重复偏好ID拒绝", lambda: collate_pairs([rows[0],rows[0]],codec,32))
    rejects("相同回答不配对", lambda: collate_pairs([dict(rows[0],rejected="2")],codec,32))
    logits = torch.randn(4, batch["labels"].shape[1], len(codec.tokens), dtype=torch.float64, requires_grad=True)
    sums, counts = sequence_logps(logits, batch["labels"])
    manual = []
    for i in range(4):
        terms = [F.log_softmax(logits[i,t-1], -1)[batch["labels"][i,t]]
                 for t in range(1, logits.shape[1]) if batch["labels"][i,t] != IGNORE]
        manual.append(torch.stack(terms).sum())
    check("logp与逐位置手算一致", torch.allclose(sums, torch.stack(manual), atol=1e-12, rtol=1e-12))
    check("有效数量包含EOS", counts.tolist() == [2,3,3,3])
    (-sums.sum()).backward()
    valid = batch["labels"][:,1:].ne(IGNORE)
    check("无监督预测位置直接梯度为0", torch.equal(logits.grad[:,:-1][~valid], torch.zeros_like(logits.grad[:,:-1][~valid])))
    check("末位置logits不参与损失", logits.grad[:,-1].abs().max() == 0)
    check("有监督位置梯度非零", logits.grad[:,:-1][valid].abs().sum() > 0)
    extra_logits = torch.cat([logits.detach(), torch.randn(4,3,len(codec.tokens),dtype=torch.float64)],1)
    extra_labels = torch.cat([batch["labels"], torch.full((4,3),IGNORE,dtype=torch.long)],1)
    check("追加PAD不改变序列logp", torch.allclose(sequence_logps(extra_logits,extra_labels)[0],sums.detach()))
    uniform = torch.zeros_like(logits)
    uniform_sum, uniform_count = sequence_logps(uniform,batch["labels"])
    check("序列使用求和不是均值", torch.allclose(uniform_sum, -uniform_count.double()*math.log(len(codec.tokens))))
    all_ignore = torch.full_like(batch["labels"],IGNORE)
    rejects("全忽略标签拒绝", lambda: sequence_logps(logits,all_ignore))
    bad = batch["labels"].clone();bad[0,-1] = len(codec.tokens)
    rejects("标签越界拒绝", lambda: sequence_logps(logits,bad))
    bad = batch["labels"].clone();bad[0,0] = BOS
    rejects("首位置标签拒绝", lambda: sequence_logps(logits,bad))
    extreme = torch.tensor([[[1000.0,-1000.0],[1000.0,-1000.0]]],dtype=torch.float64)
    check("极端logits仍有限", math.isfinite(sequence_logps(extreme,torch.tensor([[IGNORE,1]]))[0].item()))
    pc=torch.tensor([-2.,-3.],dtype=torch.float64,requires_grad=True)
    pr=torch.tensor([-4.,-2.],dtype=torch.float64,requires_grad=True)
    rc=pc.detach().clone().requires_grad_();rr=pr.detach().clone().requires_grad_()
    losses=dpo_losses(pc,pr,rc,rr,0.2)
    check("初始loss为log2", torch.allclose(losses,torch.full_like(pc,math.log(2))))
    losses.mean().backward()
    check("chosen score梯度方向", bool((pc.grad<0).all()))
    check("rejected score梯度方向", bool((pr.grad>0).all()))
    check("reference不接收梯度", rc.grad is None and rr.grad is None)
    check("相对margin增加降低loss", dpo_losses(pc.detach()+1,pr.detach(),rc,rr).mean() < math.log(2))
    check("共同平移不改变DPO", torch.allclose(dpo_losses(pc.detach()+5,pr.detach()+5,rc,rr),dpo_losses(pc.detach(),pr.detach(),rc,rr)))
    check("参考差值不能省略", not torch.allclose(dpo_losses(pc.detach(),pr.detach(),rc+2,rr),dpo_losses(pc.detach(),pr.detach(),rc,rr)))
    rejects("beta0拒绝", lambda:dpo_losses(pc,pr,rc,rr,0))
    rejects("betaNaN拒绝", lambda:dpo_losses(pc,pr,rc,rr,float("nan")))
    rejects("logp形状错位拒绝", lambda:dpo_losses(pc,pr[:1],rc,rr))
    mech=mechanisms()
    check("共享计算图bug零梯度", mech["shared_graph_bug"]["chosen_gradient"]==0)
    check("chosen概率下降也能降loss", mech["chosen_probability_can_drop"]["loss_after"] < mech["chosen_probability_can_drop"]["loss_before"])
    mc=ModelConfig(len(codec.tokens),dim=16,heads=4,layers=2,ff_dim=32,max_len=32)
    base=TinyLM(mc).double().eval()
    input_ids=batch["input_ids"]
    out=base(input_ids)
    check("模型输出形状", out.shape==(*input_ids.shape,len(codec.tokens)))
    modified=input_ids[:1].clone();modified[0,-1]=codec.to_id["7"]
    # 单独取无padding序列验证因果性。
    clean=torch.tensor([[BOS]+codec.encode("last:1,2=")])
    changed=clean.clone();changed[0,-1]=codec.to_id["3"]
    check("未来不影响前缀", torch.allclose(base(clean)[:,:-1],base(changed)[:,:-1],atol=1e-12,rtol=1e-12))
    padded=F.pad(clean,(0,3),value=PAD)
    check("右补齐不改变有效logits", torch.allclose(base(clean),base(padded)[:,:clean.shape[1]],atol=1e-12,rtol=1e-12))
    reference=freeze_reference(base)
    policy=inject_lora(copy.deepcopy(base),rank=2,alpha=4).double()
    opt=torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad],lr=0.01,weight_decay=0)
    audit_training_objects(policy,reference,opt)
    check("初始注入输出不变", torch.equal(policy(clean),reference(clean)))
    check("参考与策略存储独立", all(a.data_ptr()!=b.data_ptr() for a,b in zip(base.parameters(),reference.parameters())))
    check("正确LoRA参数量", sum(p.numel() for n,p in policy.named_parameters() if is_adapter(n))==2*2*2*2*16)
    rejects("拒绝二次注入",lambda:inject_lora(policy))
    wrongopt=torch.optim.AdamW(base.parameters(),lr=0.01)
    rejects("拒绝旧优化器",lambda:audit_training_objects(policy,reference,wrongopt))
    ref_hash=tensor_hash(base_state(reference));pol_hash=tensor_hash(base_state(policy))
    state0=adapter_snapshot(policy)
    one_dpo_step(policy,reference,batch,opt,beta=0.1)
    check("首步A梯度为0", all(p.grad.abs().max()==0 for n,p in policy.named_parameters() if n.endswith("lora_A")))
    check("首步B有非零梯度", sum(p.grad.abs().sum() for n,p in policy.named_parameters() if n.endswith("lora_B"))>0)
    check("基座未收到新梯度", all(p.grad is None for n,p in policy.named_parameters() if not is_adapter(n)))
    check("adapter确实更新", tensor_hash(adapter_snapshot(policy))!=tensor_hash(state0))
    one_dpo_step(policy,reference,batch,opt,beta=0.1)
    check("第二步A收到非零梯度", sum(p.grad.abs().sum() for n,p in policy.named_parameters() if n.endswith("lora_A"))>0)
    check("reference权重未变", tensor_hash(base_state(reference))==ref_hash)
    check("policy基座未变",tensor_hash(base_state(policy))==pol_hash)
    check("reference无梯度",all(p.grad is None for p in reference.parameters()))
    with adapter_disabled(policy):
        check("关闭新增adapter回到参考",torch.equal(policy(clean),reference(clean)))
    check("adapter模式恢复",all(m.enabled for m in policy.modules() if isinstance(m,LoRALinear)))
    cloned=copy.deepcopy(policy); saved=adapter_snapshot(policy)
    with torch.no_grad():
        next(p for n,p in policy.named_parameters() if is_adapter(n)).add_(1)
    check("最佳快照不别名",tensor_hash(saved)==tensor_hash(adapter_snapshot(cloned)))
    load_adapter(policy,saved)
    check("重载adapter恢复输出",torch.equal(policy(clean),cloned(clean)))
    cache=precompute_reference(reference,rows,codec,2)
    c,r=cached_pair_logps(cache,rows,cache_contract(reference,codec),torch.device("cpu"),torch.float64)
    with torch.no_grad():
        cc,rrr,_=pair_logps(reference,batch)
    check("缓存等于在线参考",torch.allclose(c,cc,atol=1e-12) and torch.allclose(r,rrr,atol=1e-12))
    with torch.no_grad():
        independent_chosen, independent_rejected = [], []
        for row in rows:
            single = collate_pairs([row], codec, 32)
            cc1, rr1, _ = pair_logps(reference, single)
            independent_chosen.append(cc1[0]); independent_rejected.append(rr1[0])
    check("合并批次chosen与逐题对齐", torch.allclose(cc, torch.stack(independent_chosen), atol=1e-12, rtol=1e-12))
    check("合并批次rejected与逐题对齐", torch.allclose(rrr, torch.stack(independent_rejected), atol=1e-12, rtol=1e-12))
    reversed_rows=list(reversed(rows))
    c2,r2=cached_pair_logps(cache,reversed_rows,cache_contract(reference,codec),torch.device("cpu"),torch.float64)
    check("缓存按ID而不是行号匹配",torch.allclose(c2,c.flip(0)))
    wrong=copy.deepcopy(rows);wrong[0]["chosen"]="3"
    rejects("回答变动缓存失效",lambda:cached_pair_logps(cache,wrong,cache_contract(reference,codec),torch.device("cpu"),torch.float64))
    bad_contract=copy.deepcopy(cache["contract"]);bad_contract["reference_hash"]="wrong"
    rejects("参考身份变化缓存失效",lambda:cached_pair_logps(cache,rows,bad_contract,torch.device("cpu"),torch.float64))
    p1,p2=copy.deepcopy(cloned),copy.deepcopy(cloned)
    o1=torch.optim.SGD([p for p in p1.parameters() if p.requires_grad],lr=0.01)
    o2=torch.optim.SGD([p for p in p2.parameters() if p.requires_grad],lr=0.01)
    one_dpo_step(p1,reference,batch,o1,0.1)
    one_dpo_step(p2,reference,batch,o2,0.1,ref_values=(c,r))
    check("缓存与在线一步更新等价",all(torch.allclose(x,y,atol=1e-12,rtol=1e-12)
                                   for x,y in zip(p1.parameters(),p2.parameters())))
    stats1=evaluate_pairs(cloned,reference,rows,codec,0.1,batch_size=1)
    stats2=evaluate_pairs(cloned,reference,rows,codec,0.1,batch_size=2)
    check("评测按pair加权不随batch改变",abs(stats1["dpo_loss"]-stats2["dpo_loss"])<1e-12)
    mode=cloned.training; hash_before=tensor_hash(cloned.state_dict())
    generation=greedy_generate(cloned,codec,"last:1,2=",max_new_tokens=2)
    check("生成有预算与停止原因",len(generation["new_ids"])<=2 and generation["stop_reason"] in {"eos","max_new_tokens","context_limit"})
    check("生成恢复模型模式",cloned.training==mode)
    check("生成不更新参数",tensor_hash(cloned.state_dict())==hash_before)
    check("生成不包含PAD/BOS",all(i not in {PAD,BOS} for i in generation["new_ids"]))
    with tempfile.TemporaryDirectory() as td:
        path=Path(td)/"ad.pt";torch.save(saved,path)
        recovered=torch.load(path,map_location="cpu",weights_only=True)
        check("adapter张量落盘重载",tensor_hash(recovered)==tensor_hash(saved))
    return {"passed":len(names),"checks":names,"device":"cpu","precision":"float64 mechanisms",
            "scope":"真实小型因果网络与LoRA检查，非预训练模型/CUDA测量"}


def exercise_checks_a(fn: Callable) -> dict:
    """fn(logits, labels)->每条回答logp和。"""
    seed_all(581)
    x=torch.randn(2,6,7,dtype=torch.float64,requires_grad=True)
    labels=torch.tensor([[IGNORE,IGNORE,2,3,EOS,IGNORE],[IGNORE,4,5,3,1,EOS]])
    got=fn(x,labels)
    expected=sequence_logps(x,labels)[0]
    torch.testing.assert_close(got,expected,rtol=1e-12,atol=1e-12)
    g1=torch.autograd.grad(got.sum(),x,retain_graph=True)[0]
    g2=torch.autograd.grad(expected.sum(),x)[0]
    torch.testing.assert_close(g1,g2,rtol=1e-12,atol=1e-12)
    return {"passed":True,"checks":["数值","梯度","提示和PAD忽略","只移位一次","包含EOS"]}


def exercise_checks_b(fn: Callable) -> dict:
    """fn(pc,pr,rc,rr,beta)->逐pair loss。"""
    pc=torch.tensor([-2.,-5.],dtype=torch.float64,requires_grad=True)
    pr=torch.tensor([-3.,-4.],dtype=torch.float64,requires_grad=True)
    rc=torch.tensor([-1.,-6.],dtype=torch.float64,requires_grad=True)
    rr=torch.tensor([-4.,-2.],dtype=torch.float64,requires_grad=True)
    got=fn(pc,pr,rc,rr,0.3)
    expected=dpo_losses(pc,pr,rc,rr,0.3)
    torch.testing.assert_close(got,expected,rtol=1e-12,atol=1e-12)
    got.mean().backward()
    require(pc.grad is not None and pr.grad is not None,"策略梯度被切断")
    require(rc.grad is None and rr.grad is None,"参考收到了梯度")
    return {"passed":True,"checks":["方向","beta","参考差值","参考冻结","策略梯度"]}


def exercise_checks_c(fn: Callable) -> dict:
    """fn(policy,reference,batch,optimizer,beta) 完成一次标准DPO更新，裁剪阈值1。"""
    seed_all(583)
    codec=CharCodec();rows=make_splits()["train"][:3]
    base=TinyLM(ModelConfig(len(codec.tokens),dim=16,heads=4,layers=1,ff_dim=32)).double()
    ref=freeze_reference(base)
    p1=inject_lora(copy.deepcopy(base),2,4.0);p2=copy.deepcopy(p1)
    o1=torch.optim.SGD([p for p in p1.parameters() if p.requires_grad],lr=0.05)
    o2=torch.optim.SGD([p for p in p2.parameters() if p.requires_grad],lr=0.05)
    batch=collate_pairs(rows,codec,32)
    for m in [p1,p2]:
        for p in m.parameters():
            if p.requires_grad:p.grad=torch.ones_like(p)  # 检查是否清理旧梯度。
    ref_hash=tensor_hash(ref.state_dict());base_hash=tensor_hash(base_state(p1))
    fn(p1,ref,batch,o1,0.2)
    one_dpo_step(p2,ref,batch,o2,0.2)
    for a,b in zip(p1.parameters(),p2.parameters()):
        torch.testing.assert_close(a,b,rtol=1e-12,atol=1e-12)
    require(tensor_hash(base_state(p1))==base_hash,"改动了冻结基座")
    require(tensor_hash(ref.state_dict())==ref_hash and all(p.grad is None for p in ref.parameters()),"参考改变")
    return {"passed":True,"checks":["清理旧梯度","pair平均","梯度裁剪","仅更新adapter","参考不变"]}

In [25]:
check_report = run_checks()
print("固定检查通过:", check_report["passed"])
print("最后六项:")
for name in check_report["checks"][-6:]:
    print("✓", name)
if RUN_DIR is not None:
    write_json(RUN_DIR / "checks.json", check_report)

固定检查通过: 74
最后六项:
✓ 评测按pair加权不随batch改变
✓ 生成有预算与停止原因
✓ 生成恢复模型模式
✓ 生成不更新参数
✓ 生成不包含PAD/BOS
✓ adapter张量落盘重载


## 14. 独立练习 A：回答序列 log 概率

实现 `completion_logps_ex(logits, labels)`，返回 `[B]` 的回答logp和。

要求：只移位一次，屏蔽提示/PAD，保留EOS，使用数值稳定的实现和安全下标，保留到策略logits的梯度。先不要调用本课已经实现的 `sequence_logps()`。

In [26]:
def completion_logps_ex(logits, labels):
    raise NotImplementedError("由你实现：返回逐条completion的logp和")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    print(exercise_checks_a(completion_logps_ex))
else:
    print("练习A尚未验收：先补全函数，再打开开关。")

练习A尚未验收：先补全函数，再打开开关。


## 15. 独立练习 B：DPO loss

实现 `dpo_loss_ex(pc, pr, rc, rr, beta)`，返回每对的loss，不在函数内部求batch均值。

核对：chosen/rejected方向、beta、参考差值、参考不收梯度、当前策略收梯度。不要直接调用已实现的 `dpo_losses()`。

In [27]:
def dpo_loss_ex(pc, pr, rc, rr, beta):
    raise NotImplementedError("由你实现：逐pair的标准sigmoid-DPO")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    print(exercise_checks_b(dpo_loss_ex))
else:
    print("练习B尚未验收。")

练习B尚未验收。


## 16. 独立练习 C：一次真正的 LoRA 更新

实现 `dpo_update_ex(policy, reference, batch, optimizer, beta)`。

可以复用本课的 `pair_logps()` 与 `dpo_losses()`，但自己组织清零、参考no_grad、pair均值、反传、阈值1的梯度裁剪和更新。验收函数会故意预填旧梯度，检测是否清理，并比较基座和参考是否保持不变。

In [28]:
def dpo_update_ex(policy, reference, batch, optimizer, beta):
    raise NotImplementedError("由你实现完整的一次adapter更新")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    print(exercise_checks_c(dpo_update_ex))
else:
    print("练习C尚未验收。")

练习C尚未验收。


---
## 17. 参考答案：独立完成后再核对

以下函数使用不同名字，不会覆盖练习区。短答案专注核心步骤，完整脚本中的正式函数还含输入类型、有限性和身份校验。

In [29]:
def completion_logps_solution(logits, labels):
    targets = labels[:, 1:]
    valid = targets != IGNORE
    safe = targets.masked_fill(~valid, 0)
    prediction = logits[:, :-1, :]
    if prediction.dtype in (torch.float16, torch.bfloat16):
        prediction = prediction.float()
    values = F.log_softmax(prediction, -1).gather(-1, safe.unsqueeze(-1)).squeeze(-1)
    return values.masked_fill(~valid, 0).sum(-1)

In [30]:
def dpo_loss_solution(pc, pr, rc, rr, beta):
    policy_ratio = pc - pr
    reference_ratio = rc.detach() - rr.detach()
    return F.softplus(-beta * (policy_ratio - reference_ratio))

In [31]:
def dpo_update_solution(policy, reference, batch, optimizer, beta):
    policy.train()
    reference.eval()
    optimizer.zero_grad(set_to_none=True)
    pc, pr, _ = pair_logps(policy, batch)
    with torch.no_grad():
        rc, rr, _ = pair_logps(reference, batch)
    loss = dpo_losses(pc, pr, rc, rr, beta).mean()
    loss.backward()
    nn.utils.clip_grad_norm_([p for p in policy.parameters() if p.requires_grad],
                            1.0, error_if_nonfinite=True)
    optimizer.step()
    return loss.item()

In [32]:
reference_solution_checks = {
    "A": exercise_checks_a(completion_logps_solution),
    "B": exercise_checks_b(dpo_loss_solution),
    "C": exercise_checks_c(dpo_update_solution),
}
print(json.dumps(reference_solution_checks, ensure_ascii=False, indent=2))

{
  "A": {
    "passed": true,
    "checks": [
      "数值",
      "梯度",
      "提示和PAD忽略",
      "只移位一次",
      "包含EOS"
    ]
  },
  "B": {
    "passed": true,
    "checks": [
      "方向",
      "beta",
      "参考差值",
      "参考冻结",
      "策略梯度"
    ]
  },
  "C": {
    "passed": true,
    "checks": [
      "清理旧梯度",
      "pair平均",
      "梯度裁剪",
      "仅更新adapter",
      "参考不变"
    ]
  }
}


## 18. 本课验收

能够解释以下四点，再进入下一课：

1. 为什么同一个偏好batch要形成2B条序列？哪一侧需要计算图？
2. 为什么初始loss接近log2，而参考可能已经更偏好chosen或rejected？
3. 为什么“参考校正margin变好”与“自由生成答案正确”并不等价？
4. 参考来自SFT adapter时，直接关闭所有adapter为什么可能选错锚点？

### 资料

- [DPO 原论文](https://arxiv.org/abs/2305.18290)：固定参考的偏好目标与推导。
- [论文作者实现](https://github.com/eric-mitchell/direct-preference-optimization)：参考与策略的概率计算。
- [TRL DPO 文档](https://huggingface.co/docs/trl/dpo_trainer)：成对训练、参考管理与预计算。
- [LoRA 原论文](https://arxiv.org/abs/2106.09685)：冻结权重与低秩增量。
- [PyTorch 自动求导](https://docs.pytorch.org/docs/2.10/notes/autograd.html)与[log_softmax](https://docs.pytorch.org/docs/2.10/generated/torch.nn.functional.log_softmax.html)：梯度模式和稳定概率计算。

**下一课：第59课｜可验证奖励与策略梯度——REINFORCE、Baseline与KL约束。** 从固定偏好对走向按当前策略采样，继续区分奖励数字、梯度估计与真实任务效果。